# Defensive Web Application Security & API Architecture LLM Pipeline

Bu notebook, **Google Colab** (hem **TPU v5e-1** PyTorch-XLA/Optimum-TPU hem de **CUDA A100/L4** GPU) ortamlarına otomatik uyum sağlayan hibrit eğitim, veri fabrikası ve Best-of-N değerlendirme boru hattını sırasıyla çalıştırır.

**Güvenlik Kapsamı:** Yalnızca savunma odaklı mimari analiz, zafiyet tespiti, yanlış pozitif (false positive) eleme ve sunucu tarafı iyileştirme (remediation) kodu üretimi içerir.

## 1. Ortam Kurulumu ve Donanım Algılama (TPU v5e-1 / CUDA A100 / L4)

In [ ]:
# Colab Runtime Detection & Dependency Installation
import os, sys, subprocess

IS_TPU = bool(
    os.environ.get("COLAB_TPU_ADDR")
    or os.environ.get("TPU_NAME")
    or os.environ.get("PJRT_DEVICE", "").upper() == "TPU"
    or os.environ.get("TPU_ACCELERATOR_TYPE")
)
print(f"Detected TPU env: {IS_TPU}")

if not os.path.exists("requirements.txt"):
    raise FileNotFoundError("Proje kök dizininde olduğunuzdan emin olun (requirements.txt bulunamadı).")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
if IS_TPU:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_xla[tpu]", "optimum-tpu", "-f", "https://storage.googleapis.com/libtpu-releases/index.html"])

from src.common.hardware import detect_hardware_profile
hw = detect_hardware_profile("configs/hardware_profiles.yaml")
print(f"Selected Profile: {hw.profile_name} | Backend: {hw.backend} | Device: {hw.device_name}")

## 2. Veri Fabrikası (Ingestion -> MinHash LSH Dedup & 40% SAFE Balance -> Reasoning Distillation -> 4096-Token XLA Packing)

In [ ]:
# Step 2.1: Multi-source ingestion (OSV.dev, HF PrimeVul/MegaVul/DPO, Semgrep rules, APIs-guru OpenAPI)
!python -m src.data.ingest_sources --output data/raw/ingested_samples.jsonl --hf-max-rows 300

# Step 2.2: Synthesize multi-turn black-box HTTP audit scenarios (Turn 1 baseline vs Turn 2 diff, 40% SAFE)
!python -m src.data.synthesize_blackbox --input data/raw/ingested_samples.jsonl --output data/raw/blackbox_synthesized.jsonl --safe-ratio 0.40

# Merge raw + synthesized black-box scenarios
import pathlib
merged = pathlib.Path("data/raw/merged_samples.jsonl")
merged.write_text(
    pathlib.Path("data/raw/ingested_samples.jsonl").read_text(encoding="utf-8")
    + pathlib.Path("data/raw/blackbox_synthesized.jsonl").read_text(encoding="utf-8"),
    encoding="utf-8",
)

# Step 2.3: MinHash LSH (threshold=0.85) deduplication & 40% hardened SAFE balancing across 3 experts
!python -m src.data.dedup_and_balance --input data/raw/merged_samples.jsonl --output-dir data/balanced --threshold 0.85 --safe-ratio 0.40

# Step 2.4: Dual-mode <think>...</think> + strict JSON (with verification_assertion) distillation
!python -m src.data.distill_reasoning --input-dir data/balanced --output-dir data/distilled

# Step 2.5: ChatML tokenization & zero-padding 4096-token packing to /content/data_packed
!python -m src.data.pack_sequences --input-dir data/distilled --output-dir /content/data_packed --config configs/hardware_profiles.yaml

## 3. 3 Uzman LoRA Eğitimi (Auth/BOLA, Injection/Validation, Logic Verifier)

In [ ]:
EXPERTS = [
    "adapter_auth_bola",
    "adapter_injection_validation",
    "adapter_logic_verifier",
]

for expert in EXPERTS:
    print(f"=== Training Specialist LoRA (SFT): {expert} ===")
    !python -m src.train.train_expert_lora \
        --expert {expert} \
        --packed-dir /content/data_packed \
        --output-dir /content/outputs/adapters \
        --config configs/hardware_profiles.yaml

print("=== Post-SFT DPO False-Positive Elimination: adapter_logic_verifier ===")
!python -m src.train.train_verifier_dpo \
    --distilled-dir data/distilled \
    --sft-adapter-dir /content/outputs/adapters/adapter_logic_verifier \
    --output-dir /content/outputs/adapters/adapter_logic_verifier_dpo \
    --config configs/hardware_profiles.yaml

## 4. Best-of-N Çıkarım ve Benchmark Değerlendirmesi

In [ ]:
# Run Multi-Adapter Best-of-N (N=4) Benchmark Suite
!python -m src.eval.benchmark_runner \
    --adapters-dir /content/outputs/adapters \
    --best-of-n 4 \
    --config configs/hardware_profiles.yaml \
    --report-out /content/outputs/benchmark_report.json

import json
with open("/content/outputs/benchmark_report.json", "r", encoding="utf-8") as f:
    report = json.load(f)
print(json.dumps(report["metrics"], indent=2))